In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [3]:
# The more complex system becomes, ie the more context, the more specialisms, and the more tools a system requires, the higher the likelyhood that a single agent will get overloaded
# To deal with this, we can have a single agent orchestrate subagents!
# The subagents are essentially tools, for the main (orchestrator) agent to call

from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [4]:
from langchain.agents import create_agent

# create subagents
# these subagents use the tools, ie their specialisms

subagent_1 = create_agent(
    model='ollama:gemma4',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='ollama:gemma4',
    tools=[square]
)

## Calling subagents

In [5]:
from langchain.messages import HumanMessage

# So we are literally writing tools, where the function code invokes the subagents...
@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='ollama:gemma4',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [8]:
question = "What is the square of 5?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [9]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square of 5?', additional_kwargs={}, response_metadata={}, id='8f000b1d-8894-47e0-8079-47472ce95100'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-10-15T20:42:53.888816Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2753222250, 'load_duration': 4661042, 'prompt_eval_count': 154, 'prompt_eval_duration': 486227000, 'eval_count': 62, 'eval_duration': 2253891000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, id='lc_run--01a1414d-a97d-71c0-9ce8-e2f51422a3ca-0', tool_calls=[{'name': 'call_subagent_2', 'args': {'x': 5}, 'id': 'cf599c49-dd30-4c47-9a73-26d7dc8d4262', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 154, 'output_tokens': 62, 'total_tokens': 216}),
              ToolMessage(content='25.0', name='call_subagent_2', id='d887e61d-e126-4d1b-8a2b-29208cecd0fb', tool_call_id='cf599c49-dd30-4c47-9a73-26d7d